In [5]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2025, 12, 29, 7, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 17, 30))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

CACHE HIT...: 100%|██████████| 1/1 [00:00<00:00, 48.39it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name
0,1570925913000000501,1.520765e+18,CORR,,2025-12-29 12:00:00+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
1,1570925913000000201,1.493091e+18,CORR,,2025-12-29 12:00:00+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
2,1570925913000000101,1.493025e+18,CORR,,2025-12-29 12:00:00+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
3,1570925913000000301,1.495641e+18,CORR,,2025-12-29 12:00:00+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
4,1570925913000000601,1.523207e+18,CORR,,2025-12-29 12:00:00+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7214,1573822164000000101,1.571907e+18,CORR,,2025-12-29 22:25:20+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZH55DK310GD,NA/Swap OIS USD,USD-SOFR-OIS Compound
7215,1573822391000000201,1.571907e+18,TERM,NOVA,2025-12-29 22:25:35+00:00,None,IR,None,N,True,...,,NaN,,,NaN,None,None,QZH55DK310GD,NA/Swap OIS USD,USD-SOFR-OIS Compound
7216,1573836986000000101,1.460365e+18,MODI,TRAD,2025-12-29 22:25:48+00:00,True,IR,None,N,False,...,,NaN,,,NaN,None,None,QZ4L8GF8J13B,NA/Swap Fxd Fxd EUR USD,N/A
7217,1573824423000000201,NaN,NEWT,TRAD,2025-12-29 22:27:35+00:00,False,IR,None,N,False,...,,NaN,,,NaN,None,None,QZM5SJHCM72P,NA/Swap OIS USD,USD-SOFR-OIS Compound


In [9]:
from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 


USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path)

# from SDRUtils.products.usd.usd_swaptions import USD_Swaptions

# sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
# sdf

CACHE HIT...: 100%|██████████| 1/1 [00:00<00:00, 52.06it/s]


,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,matched_ust_maturity_trade_confidence,invoice_swap_ticker,is_mac,is_spreadover,is_asset_swap,risk
0,CORR-,1572601173000000201,2025-12-29 12:05:58+00:00,2026-03-18,2046-03-18 00:00:00,OIS_SWAP,IMM_H2026 IMM_H2046,250000000.0,USD,True,...,NaN,None,None,2046-03-18,NaN,None,False,False,False,342200.0
1,CORR-,1570357320000000101,2025-12-29 12:11:58+00:00,2036-03-19,2046-03-19 00:00:00,OIS_SWAP,IMM_H2036 IMM_H2046,46000000.0,USD,False,...,NaN,None,None,2046-03-19,NaN,None,False,False,False,24700.0
2,CORR-,1570353280000000201,2025-12-29 12:11:58+00:00,2046-03-21,2056-03-21 00:00:00,OIS_SWAP,IMM_H2046 IMM_H2056,71000000.0,USD,False,...,NaN,None,None,2056-03-21,NaN,None,False,False,False,24500.0
3,TERM-ETRM,1570834489000000301,2025-12-29 12:09:34+00:00,2026-03-18,2051-03-18 00:00:00,OIS_SWAP,IMM_H2026 IMM_H2051,250000000.0,USD,True,...,NaN,None,None,2051-03-18,NaN,None,False,False,False,389700.0
4,MODI-TRAD,1571015129000000201,2025-12-29 12:56:21+00:00,2026-03-31,2044-05-16 00:00:00,OIS_SWAP,3M 18Y,31000000.0,USD,False,...,NaN,None,None,2044-05-16,NaN,None,False,False,False,39800.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1900,NEWT-TRAD,1573656731000000101,2025-12-29 21:54:42+00:00,2025-12-31,2026-12-31 00:00:00,OIS_SWAP,spot 1Y,41000000.0,USD,False,...,91282CME8,2-Year,2024-12-31,2026-12-31,low,NaN,False,False,False,4000.0
1901,NEWT-TRAD,1573666981000000101 / 1573660614000000101,2025-12-29T21:55:48+00:00 / 2025-12-29T21:55:3...,2025-12-31,2035-12-31T00:00:00 / 2055-12-31T00:00:00,OIS_SWAP,10Y / 30Y,5e+07 / 2.5e+07,USD,False,...,NaN,NaN,NaN,2035-12-31 / 2055-12-31,NaN,NaN,False,False,False,41900.0
1902,NEWT-TRAD,1573691775000000101,2025-12-29 21:59:26+00:00,2025-12-31,2032-12-31 00:00:00,OIS_SWAP,spot 7Y,4000000.0,USD,False,...,91282CPQ8,7-Year,2025-12-31,2032-12-31,low,NaN,False,False,False,2500.0
1903,NEWT-TRAD,1573723140000000101,2025-12-29 21:56:25+00:00,2025-12-31,2035-11-15 00:00:00,OIS_SWAP,spot 10Y,120000000.0,USD,False,...,91282CPJ4,10-Year,2025-11-17,2035-11-15,high,NaN,False,False,False,99600.0


In [43]:
temp = df[df["Dissemination Identifier"].isin(sdf["trade_id"])]
temp["Execution Timestamp"] = temp["Execution Timestamp"].astype(str)
temp["Event timestamp"] = temp["Event timestamp"].astype(str)
temp.to_excel("swaption_trades.xlsx",index=False)

C:\Users\chris\AppData\Local\Temp\ipykernel_79052\2697270985.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\chris\AppData\Local\Temp\ipykernel_79052\2697270985.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [32]:
sdf[sdf["is_capped"] == True]

# sdf["trade_label"].value_counts().head(10)
# sdf[sdf["trade_label"] == "USD-SOFR-OIS Compound 1D CONSTANT 9Y10Y PAYER EURO VANILLA PHYS"]


,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,estimated_pv01,...,underlying_expiration_date,tenor_years,tenor_label,forward_start_years,forward_label,premium,exercise_style,strike,is_capped,Dissemination Identifier
169,NEWT-TRAD,1570783121000001401,2025-12-29 13:52:52+00:00,2025-12-29,2028-12-28,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3Y1Y RECEIVE...,6.500000e+08,USD,0.0,...,2030-01-02,1.027778,1Y,3.041667,3Y,0.000,EUROPEAN,0.035130,True,1570783121000001401
168,NEWT-TRAD,1570783120000001301,2025-12-29 13:52:52+00:00,2025-12-29,2028-12-28,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3Y1Y RECEIVE...,6.500000e+08,USD,0.0,...,2030-01-02,1.027778,1Y,3.041667,3Y,7540000.000,EUROPEAN,0.035130,True,1570783120000001301
202,TERM-EXER,1570979154000000501,2025-12-29 16:00:27+00:00,2025-10-28,2025-12-29,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 2M10Y PAYER ...,2.500000e+08,USD,0.0,...,2035-12-31,10.150000,10Y,0.172222,2M,0.000,EUROPEAN,0.037175,True,1570979154000000501
0,TERM-EXER,1570966806000001101,2025-12-29 16:00:29+00:00,2025-11-20,2025-12-29,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 1M10Y PAYER ...,2.500000e+08,USD,0.0,...,2035-12-31,10.150000,10Y,0.108333,1M,34375.000,EUROPEAN,0.035675,True,1570966806000001101
44,MODI-TRAD,1570991358000000101,2025-12-29 16:18:01+00:00,2025-12-29,2028-12-29,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3Y1M1Y RECEI...,6.500000e+08,USD,0.0,...,2030-01-03,1.027778,1Y,3.044444,3Y1M,7540000.000,EUROPEAN,0.035130,True,1570991358000000101
45,MODI-TRAD,1570991359000000201,2025-12-29 16:18:02+00:00,2025-12-29,2028-12-29,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3Y1M1Y RECEI...,6.500000e+08,USD,0.0,...,2030-01-03,1.027778,1Y,3.044444,3Y1M,0.000,EUROPEAN,0.035130,True,1570991359000000201
247,NEWT-TRAD,1572479164000000101,2025-12-29 18:20:49+00:00,2025-12-29,2026-03-30,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3M10Y RECEIV...,2.500000e+08,USD,0.0,...,2036-04-01,10.152778,10Y,0.252778,3M,1043753.340,EUROPEAN,0.035150,True,1572479164000000101
248,NEWT-TRAD,1572479167000000401,2025-12-29 18:20:56+00:00,2025-12-29,2026-03-30,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 3M10Y PAYER ...,2.500000e+08,USD,0.0,...,2036-04-01,10.152778,10Y,0.252778,3M,1056253.380,EUROPEAN,0.040150,True,1572479167000000401
88,CORR-,1572504398000000301,2025-12-29 18:33:43+00:00,2025-12-29,2026-03-30,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3M10Y RECEIV...,2.500000e+08,USD,0.0,...,2036-04-01,10.152778,10Y,0.252778,3M,1056253.380,EUROPEAN,0.040150,True,1572504398000000301
87,CORR-,1572504985000000101,2025-12-29 18:33:45+00:00,2025-12-29,2026-03-30,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 3M10Y PAYER ...,2.500000e+08,USD,0.0,...,2036-04-01,10.152778,10Y,0.252778,3M,1043753.340,EUROPEAN,0.035150,True,1572504985000000101


In [31]:
sdf.iloc[0].to_dict()

{'event_action': 'NEWT-TRAD',
 'trade_id': 1568561409000000201,
 'execution_timestamp': Timestamp('2025-12-29 09:17:55+0000', tz='UTC'),
 'effective_date': Timestamp('2025-12-29 00:00:00'),
 'expiration_date': Timestamp('2026-12-29 00:00:00'),
 'product_type': 'UNKNOWN',
 'trade_label': 'USD-SOFR-OIS Compound 1D CONSTANT 1Y6Y CHOOSER EURO VANILLA ELECT AT EXERCISE',
 'notional': 10000000.0,
 'notional_currency': 'USD',
 'estimated_pv01': 0.0,
 'package_type': 'SWAPTION',
 'package_id': None,
 'package_legs': None,
 'underlying_expiration_date': Timestamp('2032-12-31 00:00:00'),
 'tenor_years': 6.094444444444444,
 'tenor_label': '6Y',
 'forward_start_years': 1.0138888888888888,
 'forward_label': '1Y',
 'premium': 164810.0,
 'exercise_style': 'EUROPEAN',
 'strike': 0.03576,
 'is_capped': False,
 'Dissemination Identifier': 1568561409000000201}

In [30]:
df[df["Dissemination Identifier"] == 1572501480000000201].iloc[0].to_dict()

{'Dissemination Identifier': 1572501480000000201,
 'Original Dissemination Identifier': 1725643037.0,
 'Action type': 'TERM',
 'Event type': 'ETRM',
 'Event timestamp': Timestamp('2025-12-29 18:30:28+0000', tz='UTC'),
 'Amendment indicator': None,
 'Asset Class': 'IR',
 'Product name': None,
 'Cleared': 'N',
 'Mandatory clearing indicator': False,
 'Execution Timestamp': Timestamp('2025-09-16 14:54:31+0000', tz='UTC'),
 'Effective Date': Timestamp('2025-09-16 00:00:00'),
 'Expiration Date': Timestamp('2026-06-24 00:00:00'),
 'Maturity date of the underlier': datetime.date(2028, 6, 26),
 'Non-standardized term indicator': False,
 'Platform identifier': 'BILT',
 'Prime brokerage transaction indicator': False,
 'Block trade election indicator': False,
 'Large notional off-facility swap election indicator': True,
 'Notional amount-Leg 1': '5',
 'Notional amount-Leg 2': '5',
 'Notional currency-Leg 1': 'USD',
 'Notional currency-Leg 2': 'USD',
 'Notional quantity-Leg 1': None,
 'Notional qu

In [ ]:
# from SDRUtils.products._swaptions.upi import make_swaption_desc_func, _build_upi_df

# swaption_upis = _build_upi_df()

# mask = df["Unique Product Identifier"].isin(swaption_upis["swaption_Identifier_UPI"])
# swaption_trades_df = df.loc[mask].copy()

# desc_fn = make_swaption_desc_func()

# swaption_trades_df.loc[:, "description"] = swaption_trades_df.apply(desc_fn, axis=1)
# swaption_trades_df

In [11]:
temp = sdf 
# temp["Execution sf"] = temp["Execution Timestamp"].astype(str)
# temp["Event timestamp"] = temp["Event timestamp"].astype(str)
temp["execution_timestamp"] = temp["execution_timestamp"].astype(str)
temp.to_excel("swaption_trades2.xlsx",index=False)

In [ ]:
swaption QZZGWPNBF5R3 USD CALL Euro Vanilla Phys, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys
swaption QZNLQ8T0N0SX USD PUTO Euro Vanilla Phys, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys

swaption QZMMWR8JKZQ8 USD PUTO Euro Vanilla Phys, underlying QZXG4P1G2KCS USD-SOFR-OIS Compound 1D Constant Phys 
swaption QZWXKVHB5F8V USD CALL Euro Vanilla Phys, underlying QZXG4P1G2KCS USD-SOFR-OIS Compound 1D Constant Phys 

swaption QZMRJ6051HQB USD OPTL Euro Vanilla Phys, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys 

swaption QZ7B7ZPS1LS5 USD PUTO Euro Vanilla Phys, underlying QZ8RQJKXHZP9 USD-SOFR-OIS Compound 1D Constant Cash

swaption QZXZSN00ZVCG USD PUTO Euro Vanilla Phys, underlying QZ1CXH05JJJH USD-SOFR-COMPOUND 1D Constant PHYS 
swaption QZXSN072GFF3 USD CALL Euro Vanilla Phys, underlying QZ1CXH05JJJH USD-SOFR-COMPOUND 1D Constant PHYS

swaption QZZLNQ2D4JQT USD CALL Euro Vanilla Phys, underlying QZKQ8QSWZKR2 USD-SOFR-OIS Compound 1Y Constant PHYS
swaption QZJ92TTHTSF0 USD PUTO Euro Vanilla Phys, underyying QZKQ8QSWZKR2 USD-SOFR-OIS Compound 1Y Constant PHYS
swaption QZHQPRHC3S7T USD OPTL Euro Vanilla Phys, underlying QZKQ8QSWZKR2 USD-SOFR-OIS Compound 1Y Constant PHYS
swaption QZJ7QTJM4H97 USD PUTO Euro Vanilla Cash, underlying QZKQ8QSWZKR2 USD-SOFR-OIS Compound 1Y Constant PHYS
swaption QZVLJBR2Z1VS USD Call Euro Vanilla Cash, underlying QZKQ8QSWZKR2 USD-SOFR-OIS Compound 1Y Constant PHYS

swaption QZKDXXXPW3X3 USD OPTL Euro Vanilla Phys, underlying QZ749DHWD023 USD-SOFR-COMPOUND 1D Constant Cash

swaption QZJ7PRWFV1G1 USD PUTO Euro Vanilla Cash, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys
swaption QZ1SC5570DC7 USD CALL Euro Vanilla Cash, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys
swaption QZJBL64PCV75 USD OPTL Euro Vanilla OPTL, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys

swaption QZ7XW3KWNFQ0 USD CALL BERM Vanilla Cash, underlying QZWVFGTL0HX0 USD-SOFR-COMPOUND 1D Constant Cash
swaption QZP4BJ4T4W9T USD CALL Euro Vanilla Cash, underlying QZWVFGTL0HX0 USD-SOFR-COMPOUND 1D Constant Cash

swaption QZC8L2PMNJZH USD PUTO Euro Vanilla Phys, underlying QZM88X1WWFMD USD-SOFR 1D Constant PHYS

swaption QZT2NCB2NRFJ USD Call Euro Vanilla Phys, underlying QZFDML7GL67S USD-LIBOR-BBA 3M Constant Cash

swaption QZRP1RVPFLQT USD PUTO Euro Vanilla Cash, underlying QZ5JTCF06XV6 USD-SOFR 1D Custom Cash
swaption QZX5JQN0TK24 USD Call Euro Vanilla Cash, underlying QZ5JTCF06XV6 USD-SOFR 1D Custom Cash

swaption QZBK1N7NJ0VF USD Call Euro Vanilla Phys, underlying QZ749DHWD023 USD-SOFR-COMPOUND 1D Constant Cash
swaption QZ2GZS235CHW USD PUTO Euro Vanilla Phys, underlying QZ749DHWD023 USD-SOFR-COMPOUND 1D Constant Cash

swaption QZTRSNZTX0F0 USD PUTO Berm Vanilla Phys, underlying QZPFD1BKD6MW USD-SOFR CME Term 1M Accreting Phys

swaption QZMC010F38PK USD PUTO Euro Vanilla Phys, underlying QZ1CMVCKP2QW USD-SOFR 3M Constant Phys
swaption QZVQ21ZLNJ5C USD OPTL Euro Vanilla Phys, underlying QZ1CMVCKP2QW USD-SOFR 3M Constant Phys
swaption QZJ03ZN7VKG7 USD Call Euro Vanilla Cash, underlying QZ1CMVCKP2QW USD-SOFR 3M Constant Phys 


In [128]:
# import datetime
# from gs_quant.data import Dataset
# from gs_quant.session import GsSession

# gs_client_id = "2eb2f48872304c1d94fa1642fa691afe"
# gs_secret_key = "91cb9c89110495d1f62d0ab0c4014555c992c2509de8f5ae2b8bf1a2d3c86bd4"
# GsSession.use(client_id=gs_client_id, client_secret=gs_secret_key, scopes=('read_product_data',))

# usd_sofr_1y10y_asset_id = "MA3YAP9YTBN0HAM4"

# start = datetime.date(2025, 1, 1)
# end = datetime.date(2025, 11, 20)

# df = Dataset("IR_SWAPTION_VOLS_V1_STANDARD").get_data(start=start, end=end, assetId=usd_sofr_1y10y_asset_id)

# df["bpvol_yr"] = df["impliedNormalVolatility"] * (252 ** (0.5))

# df

